In [ ]:
# Text classification does not need torchvision; mismatched torchvision builds can break transformers imports.
%pip uninstall -y torchvision torchaudio torchtext
%pip install -q -U "transformers>=4.48,<5" accelerate scikit-learn pandas seaborn matplotlib nltk


In [ ]:
# --- Shared setup: ELECTRA + RoBERTa hybrid experiments ---
import copy
import math
import os
import platform
import random
import re
import time
import sys
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords, wordnet
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

SPLIT_SEED = 42
TRAIN_SEEDS = [42, 123, 2024]
RANDOM_STATE = SPLIT_SEED
ELECTRA_MODEL_NAME = "google/electra-base-discriminator"
ROBERTA_MODEL_NAME = "roberta-base"
DATA_URL = "https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv"
LOCAL_DATA_PATHS = [
    Path("data/exp/promise_exp.csv"),
    Path("../data/exp/promise_exp.csv"),
]

MAX_LENGTH = 128
NUM_EPOCHS = 30
LEARNING_RATE = 2e-5
ROBERTA_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
ROBERTA_WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.10
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
ROBERTA_GRAD_ACCUM_STEPS = 8
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MIN_DELTA = 1e-4
ROBERTA_DROPOUT = 0.10
TEMPERATURE_SCALING_MAX_ITER = 50
USE_GRADIENT_CHECKPOINTING = False
BENCHMARK_WARMUP_ROUNDS = 2
BENCHMARK_REPEATS = 10

AUGMENTATION_TARGETS = {
    "FT": 200,
    "L": 200,
    "MN": 200,
    "A": 200,
    "LF": 200,
    "SC": 200,
    "PO": 200,
    "PE": 200,
    "O": 200,
    "US": 200,
    "SE": 100,
}
KEEP_WORDS = {"not", "no", "shall", "must", "should", "will", "cannot", "never", "only"}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SPLIT_SEED)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
stop_words = set(stopwords.words("english")) - KEEP_WORDS


def normalize_requirement_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def get_synonyms(word):
    clean_word = re.sub(r"[^a-zA-Z]", "", str(word).lower())
    if not clean_word:
        return []
    synonyms = set()
    for syn in wordnet.synsets(clean_word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != clean_word:
                synonyms.add(synonym)
    return list(synonyms)


def synonym_replacement(text, n=1):
    words = str(text).split()
    new_words = words.copy()
    candidates = list({
        word for word in words
        if re.sub(r"[^a-zA-Z]", "", word).lower() not in stop_words and len(word) > 3
    })
    random.shuffle(candidates)

    replaced = 0
    for word in candidates:
        synonyms = get_synonyms(word)
        if synonyms:
            synonym = random.choice(synonyms)
            new_words = [synonym if current == word else current for current in new_words]
            replaced += 1
        if replaced >= n:
            break
    return " ".join(new_words)


def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = str(text).split()
    if not words:
        return str(text)
    for _ in range(n):
        words.insert(random.randint(0, len(words) - 1), random.choice(phrases))
    return " ".join(words)


def random_swap(text, n=1):
    words = str(text).split()
    if len(words) < 2:
        return str(text)
    for _ in range(n):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return " ".join(words)


def random_deletion(text, p=0.1):
    words = str(text).split()
    if len(words) <= 1:
        return str(text)
    remaining = [word for word in words if random.random() > p]
    return " ".join(remaining) if remaining else random.choice(words)


def augment_requirement_text(text):
    text = synonym_replacement(text, n=1)
    text = phrase_insertion(text, n=1)
    text = random_swap(text, n=1)
    text = random_deletion(text, p=0.1)
    return normalize_requirement_text(text)


def load_promise_exp():
    for path in LOCAL_DATA_PATHS:
        if path.exists():
            print(f"Loaded PROMISE expanded dataset from local path: {path}")
            return pd.read_csv(path)
    print(f"Loaded PROMISE expanded dataset from URL: {DATA_URL}")
    return pd.read_csv(DATA_URL)


df = load_promise_exp()
df = df[["RequirementText", "class"]].dropna().reset_index(drop=True)
df["RequirementText"] = df["RequirementText"].astype(str).map(normalize_requirement_text)
df["class"] = df["class"].astype(str).str.strip().str.upper()
df = df[df["RequirementText"] != ""].reset_index(drop=True)
df = df[df["class"].map(df["class"].value_counts()) > 1].reset_index(drop=True)
df["model_text"] = df["RequirementText"]

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
id2label = {idx: label for idx, label in enumerate(label_encoder.classes_)}
label2id = {label: idx for idx, label in id2label.items()}
num_labels = len(label_encoder.classes_)

train_df, val_test_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=SPLIT_SEED,
)
val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.5,
    stratify=val_test_df["label"],
    random_state=SPLIT_SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Full distribution:", dict(df["class"].value_counts()))
print("Train distribution:", dict(train_df["class"].value_counts()))
print("Validation distribution:", dict(val_df["class"].value_counts()))
print("Test distribution:", dict(test_df["class"].value_counts()))
print("Labels:", list(label_encoder.classes_))
print("Using raw requirement text; no stopword/punctuation/digit removal.")


In [ ]:
# --- Dataset, metrics, and PyTorch training loop ---
electra_tokenizer = AutoTokenizer.from_pretrained(ELECTRA_MODEL_NAME, use_fast=True)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME, use_fast=True)


class ElectraRobertaDataset(Dataset):
    def __init__(self, frame):
        texts = frame["model_text"].astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.electra = electra_tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_token_type_ids=True,
        )
        self.roberta = roberta_tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "electra_input_ids": torch.tensor(self.electra["input_ids"][idx], dtype=torch.long),
            "electra_attention_mask": torch.tensor(self.electra["attention_mask"][idx], dtype=torch.long),
            "roberta_input_ids": torch.tensor(self.roberta["input_ids"][idx], dtype=torch.long),
            "roberta_attention_mask": torch.tensor(self.roberta["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }
        if "token_type_ids" in self.electra:
            item["electra_token_type_ids"] = torch.tensor(self.electra["token_type_ids"][idx], dtype=torch.long)
        return item


def make_loaders(train_frame, seed=None):
    train_ds = ElectraRobertaDataset(train_frame)
    val_ds = ElectraRobertaDataset(val_df)
    test_ds = ElectraRobertaDataset(test_df)
    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=generator)
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE)
    return train_loader, val_loader, test_loader


def compute_class_weights(frame):
    counts = frame["label"].value_counts().sort_index()
    weights = len(frame) / (len(counts) * counts)
    return torch.tensor(weights.values, dtype=torch.float)


def compute_metrics_from_arrays(labels, preds):
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": weighted_p,
        "recall": weighted_r,
        "f1": weighted_f1,
        "precision_macro": macro_p,
        "recall_macro": macro_r,
        "macro_F1": macro_f1,
    }


@torch.no_grad()
def evaluate_model(model, data_loader, device):
    model.eval()
    all_labels = []
    all_preds = []
    total_loss = 0.0

    for batch in data_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(**batch)
        total_loss += outputs["loss"].item()
        preds = outputs["logits"].argmax(dim=-1)
        all_labels.extend(batch["labels"].detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    labels = np.array(all_labels)
    preds = np.array(all_preds)
    metrics = compute_metrics_from_arrays(labels, preds)
    metrics["loss"] = total_loss / max(1, len(data_loader))
    return metrics, labels, preds


def build_optimizer(model):
    if hasattr(model, "optimizer_parameter_groups"):
        parameter_groups = model.optimizer_parameter_groups()
    else:
        parameter_groups = [
            {
                "params": [param for param in model.parameters() if param.requires_grad],
                "lr": LEARNING_RATE,
            }
        ]
    return torch.optim.AdamW(parameter_groups, weight_decay=WEIGHT_DECAY)


def train_model(model, train_loader, val_loader, run_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = build_optimizer(model)
    accumulation_steps = (
        model.gradient_accumulation_steps()
        if hasattr(model, "gradient_accumulation_steps")
        else GRAD_ACCUM_STEPS
    )
    updates_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
    total_updates = max(1, updates_per_epoch * NUM_EPOCHS)
    warmup_updates = int(total_updates * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_updates,
        num_training_steps=total_updates,
    )
    amp_enabled = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    best_state = None
    best_macro_F1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    epochs_run = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        epochs_run = epoch
        model.train()
        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, start=1):
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.amp.autocast("cuda", enabled=amp_enabled):
                outputs = model(**batch)
                loss = outputs["loss"] / accumulation_steps

            scaler.scale(loss).backward()
            total_loss += loss.item() * accumulation_steps

            if step % accumulation_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    (param for param in model.parameters() if param.requires_grad),
                    max_norm=1.0,
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_metrics, _, _ = evaluate_model(model, val_loader, device)
        train_loss = total_loss / max(1, len(train_loader))
        current_lrs = sorted({group["lr"] for group in optimizer.param_groups})
        lr_text = ",".join(f"{lr:.2e}" for lr in current_lrs)
        print(
            f"{run_name} | Epoch {epoch:02d} | train_loss={train_loss:.4f} "
            f"| val_loss={val_metrics['loss']:.4f} "
            f"| val_macro_F1={val_metrics['macro_F1']:.4f} | lr={lr_text}"
        )

        if val_metrics["macro_F1"] > best_macro_F1 + EARLY_STOPPING_MIN_DELTA:
            best_macro_F1 = val_metrics["macro_F1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(
                    f"{run_name} | early stopping at epoch {epoch}; "
                    f"best_epoch={best_epoch}, best_val_macro_F1={best_macro_F1:.4f}"
                )
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.training_metadata = {
        "best_epoch": best_epoch,
        "epochs_run": epochs_run,
        "best_val_macro_F1": best_macro_F1,
        "warmup_updates": warmup_updates,
        "maximum_optimizer_updates": total_updates,
        "gradient_accumulation_steps": accumulation_steps,
    }
    return model, device

def evaluate_and_plot(model, device, test_loader, run_name):
    metrics, labels, preds = evaluate_model(model, test_loader, device)
    print(f"{run_name} test metrics:", metrics)
    print(classification_report(
        labels,
        preds,
        target_names=label_encoder.classes_,
        zero_division=0,
    ))

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        cmap="Blues",
    )
    plt.title(f"Confusion Matrix - {run_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    return metrics


def summarize_seed_results(results_df, metric_columns):
    summary = results_df[metric_columns].agg(["mean", "std"]).T
    summary["mean +/- std"] = summary.apply(
        lambda row: f"{row['mean']:.4f} +/- {row['std']:.4f}",
        axis=1,
    )
    return summary


def run_fusion_experiments(train_frame, experiment_name, run_seed=None, run_gated=True):
    if run_seed is None:
        run_seed = RANDOM_STATE
    set_all_seeds(run_seed)
    train_loader, val_loader, test_loader = make_loaders(train_frame, seed=run_seed)
    class_weights = compute_class_weights(train_frame)
    results = []

    start_time = time.perf_counter()
    naive_model = ElectraRobertaNaiveFusion(num_labels=num_labels, class_weights=class_weights)
    naive_model = enable_memory_saving(naive_model)
    naive_model, naive_device = train_model(
        naive_model,
        train_loader,
        val_loader,
        run_name=f"ELECTRA+RoBERTa Naive Fusion ({experiment_name}, seed={run_seed})",
    )
    naive_metrics = evaluate_and_plot(
        naive_model,
        naive_device,
        test_loader,
        run_name=f"ELECTRA+RoBERTa Naive Fusion ({experiment_name}, seed={run_seed})",
    )
    naive_metrics["runtime_seconds"] = time.perf_counter() - start_time
    naive_metrics["seed"] = run_seed
    results.append({"model": f"ELECTRA+RoBERTa Naive Fusion ({experiment_name})", **naive_metrics})
    print(f"ELECTRA+RoBERTa Naive Fusion ({experiment_name}) | seed={run_seed} | runtime_seconds={naive_metrics['runtime_seconds']:.2f}")

    if run_gated:
        start_time = time.perf_counter()
        gated_model = ElectraRobertaGatedFusion(num_labels=num_labels, class_weights=class_weights)
        gated_model = enable_memory_saving(gated_model)
        gated_model, gated_device = train_model(
            gated_model,
            train_loader,
            val_loader,
            run_name=f"ELECTRA+RoBERTa Gated Fusion ({experiment_name}, seed={run_seed})",
        )
        gated_metrics = evaluate_and_plot(
            gated_model,
            gated_device,
            test_loader,
            run_name=f"ELECTRA+RoBERTa Gated Fusion ({experiment_name}, seed={run_seed})",
        )
        gated_metrics["runtime_seconds"] = time.perf_counter() - start_time
        gated_metrics["seed"] = run_seed
        results.append({"model": f"ELECTRA+RoBERTa Gated Fusion ({experiment_name})", **gated_metrics})
        print(f"ELECTRA+RoBERTa Gated Fusion ({experiment_name}) | seed={run_seed} | runtime_seconds={gated_metrics['runtime_seconds']:.2f}")

    return results


In [ ]:
# --- Frozen-selector SERF with an independently trained RoBERTa expert ---
def cls_pool(outputs):
    return outputs.last_hidden_state[:, 0, :]


def freeze_lower_encoder_layers(encoder, frozen_layer_count=8):
    # This follows the existing ELECTRA and RoBERTa baseline protocol.
    for layer in encoder.encoder.layer[:frozen_layer_count]:
        for param in layer.parameters():
            param.requires_grad = False


def set_module_trainable(module, trainable):
    for param in module.parameters():
        param.requires_grad = trainable


def report_trainable_and_frozen_parameters(model, model_name):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    frozen_params = sum(param.numel() for param in model.parameters() if not param.requires_grad)
    print(f"{model_name} trainable parameters: {trainable_params:,}")
    print(f"{model_name} frozen parameters: {frozen_params:,}")


class ElectraRobertaLateFusionSERF(nn.Module):
    """SERF with a frozen ELECTRA selector and an independent RoBERTa expert."""

    def __init__(self, num_labels, class_weights=None):
        super().__init__()
        self.electra_encoder = AutoModel.from_pretrained(ELECTRA_MODEL_NAME)
        self.roberta_encoder = AutoModel.from_pretrained(ROBERTA_MODEL_NAME)
        electra_hidden = self.electra_encoder.config.hidden_size
        roberta_hidden = self.roberta_encoder.config.hidden_size

        self.selector_head = nn.Sequential(
            nn.LayerNorm(electra_hidden),
            nn.Linear(electra_hidden, electra_hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(electra_hidden, num_labels),
        )
        # Match the independently evaluated RoBERTa baseline: dropout plus a linear head.
        self.roberta_dropout = nn.Dropout(ROBERTA_DROPOUT)
        self.roberta_head = nn.Linear(roberta_hidden, num_labels)
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else torch.ones(num_labels),
        )
        self.training_stage = "selector"
        self.training_metadata = {}
        self.configure_selector_stage(report=False)
        report_trainable_and_frozen_parameters(self, "ElectraRobertaLateFusionSERF")

    def enable_memory_saving(self):
        if USE_GRADIENT_CHECKPOINTING:
            for encoder in (self.electra_encoder, self.roberta_encoder):
                if hasattr(encoder, "gradient_checkpointing_enable"):
                    encoder.gradient_checkpointing_enable(
                        gradient_checkpointing_kwargs={"use_reentrant": False}
                    )
        return self

    def configure_selector_stage(self, report=True):
        set_module_trainable(self.electra_encoder, True)
        freeze_lower_encoder_layers(self.electra_encoder)
        set_module_trainable(self.selector_head, True)
        set_module_trainable(self.roberta_encoder, False)
        set_module_trainable(self.roberta_head, False)
        self.training_stage = "selector"
        if report:
            report_trainable_and_frozen_parameters(self, "SERF ELECTRA selector stage")

    def configure_roberta_stage(self, report=True):
        # ELECTRA remains exactly as selected by the Stage 1 validation checkpoint.
        set_module_trainable(self.electra_encoder, False)
        set_module_trainable(self.selector_head, False)
        set_module_trainable(self.roberta_encoder, True)
        freeze_lower_encoder_layers(self.roberta_encoder)
        set_module_trainable(self.roberta_head, True)
        self.training_stage = "roberta"
        if report:
            report_trainable_and_frozen_parameters(self, "SERF independent RoBERTa stage")

    def configure_inference(self):
        set_module_trainable(self, False)
        self.training_stage = "inference"
        self.eval()

    def optimizer_parameter_groups(self):
        if self.training_stage == "roberta":
            active_parameters = [
                (name, param)
                for name, param in self.named_parameters()
                if param.requires_grad
            ]
            no_decay_terms = ("bias", "LayerNorm.weight", "layer_norm.weight")
            decay_params = [
                param
                for name, param in active_parameters
                if not any(term in name for term in no_decay_terms)
            ]
            no_decay_params = [
                param
                for name, param in active_parameters
                if any(term in name for term in no_decay_terms)
            ]
            return [
                {
                    "params": decay_params,
                    "lr": ROBERTA_LEARNING_RATE,
                    "weight_decay": ROBERTA_WEIGHT_DECAY,
                },
                {
                    "params": no_decay_params,
                    "lr": ROBERTA_LEARNING_RATE,
                    "weight_decay": 0.0,
                },
            ]
        return [{
            "params": [param for param in self.parameters() if param.requires_grad],
            "lr": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
        }]

    def gradient_accumulation_steps(self):
        if self.training_stage == "roberta":
            return ROBERTA_GRAD_ACCUM_STEPS
        return GRAD_ACCUM_STEPS

    def encode_electra(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        return cls_pool(self.electra_encoder(**kwargs))

    def encode_roberta(self, input_ids, attention_mask):
        outputs = self.roberta_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        if outputs.pooler_output is not None:
            return outputs.pooler_output
        return cls_pool(outputs)

    def selector_logits_from_representation(self, h_electra):
        return self.selector_head(h_electra)

    def roberta_logits_from_representation(self, h_roberta):
        return self.roberta_head(self.roberta_dropout(h_roberta))

    def forward(
        self,
        electra_input_ids,
        electra_attention_mask,
        roberta_input_ids=None,
        roberta_attention_mask=None,
        labels=None,
        electra_token_type_ids=None,
        **kwargs,
    ):
        if self.training_stage == "selector":
            h_electra = self.encode_electra(
                electra_input_ids,
                electra_attention_mask,
                electra_token_type_ids,
            )
            logits = self.selector_logits_from_representation(h_electra)
        elif self.training_stage == "roberta":
            h_roberta = self.encode_roberta(roberta_input_ids, roberta_attention_mask)
            logits = self.roberta_logits_from_representation(h_roberta)
        else:
            raise RuntimeError("Use the explicit SERF inference functions after configure_inference().")

        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels, weight=self.class_weights)
        return {"loss": loss, "logits": logits}


In [ ]:
# --- Late-fusion SERF routing, calibration, diagnostics, and profiling ---
RISKY_CLASSES = {"A", "L", "FT", "MN", "PO", "SC", "PE", "O", "US"}
RISKY_LABEL_IDS = {label2id[label] for label in RISKY_CLASSES if label in label2id}
RISKY_POLICY_CANDIDATES = {
    "predefined": RISKY_LABEL_IDS,
    "none": set(),
}
TAU_VALUES = [0.70, 0.80, 0.85, 0.90]
DELTA_VALUES = [0.15, 0.20]
# alpha is the ELECTRA probability weight; alpha=0 uses RoBERTa on the hard path.
ALPHA_VALUES = [0.00, 0.25, 0.50, 0.75]

SERF_METRIC_COLUMNS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "precision_macro",
    "recall_macro",
    "macro_F1",
    "loss",
    "route_to_fusion_rate",
    "electra_only_rate",
    "average_active_encoder_passes",
    "mean_confidence",
    "mean_margin",
    "mean_entropy",
    "confidence_route_reason_rate",
    "margin_route_reason_rate",
    "risky_class_route_reason_rate",
    "selector_only_macro_F1",
    "roberta_only_macro_F1",
    "always_fusion_macro_F1",
    "selector_error_rate",
    "roberta_error_rate",
    "expert_disagreement_rate",
    "error_capture_rate",
    "route_precision_for_selector_errors",
    "routed_correction_rate",
    "routed_degradation_rate",
    "roberta_correction_potential",
    "late_fusion_correction_potential",
    "oracle_router_macro_F1",
    "selector_prediction_drift_rate",
    "selector_temperature",
    "roberta_temperature",
    "selector_calibration_nll_before",
    "selector_calibration_nll_after",
    "roberta_calibration_nll_before",
    "roberta_calibration_nll_after",
    "selector_epochs_run",
    "roberta_epochs_run",
    "pretokenized_latency_ms_per_sample",
    "p50_latency_ms_per_sample",
    "p95_latency_ms_per_sample",
    "throughput_samples_per_second",
    "peak_gpu_memory_mb",
    "selector_train_seconds",
    "roberta_train_seconds",
    "threshold_search_seconds",
    "routed_test_seconds",
    "runtime_seconds",
]


def collect_runtime_environment():
    environment = {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "processor": platform.processor() or "not explicitly reported by runtime",
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    }
    if hasattr(os, "sysconf") and "SC_PAGE_SIZE" in os.sysconf_names:
        total_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
        environment["system_ram_gb"] = total_bytes / (1024 ** 3)
    else:
        environment["system_ram_gb"] = None
    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        environment["gpu_name"] = torch.cuda.get_device_name(0)
        environment["gpu_memory_gb"] = properties.total_memory / (1024 ** 3)
    else:
        environment["gpu_name"] = None
        environment["gpu_memory_gb"] = None
    return environment


def synchronize_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def predicted_risky_mask(preds, risky_label_ids):
    risky_mask = torch.zeros_like(preds, dtype=torch.bool)
    for risky_id in risky_label_ids:
        risky_mask |= preds.eq(int(risky_id))
    return risky_mask


def calibrated_probabilities(logits, temperature):
    return torch.softmax(logits / max(float(temperature), 1e-6), dim=-1)


def late_fusion_probabilities(
    selector_logits,
    roberta_logits,
    selector_temperature,
    roberta_temperature,
    alpha,
):
    selector_probs = calibrated_probabilities(selector_logits, selector_temperature)
    roberta_probs = calibrated_probabilities(roberta_logits, roberta_temperature)
    return alpha * selector_probs + (1.0 - alpha) * roberta_probs


def probability_statistics(probabilities):
    top2 = torch.topk(probabilities, k=2, dim=-1).values
    confidence = top2[:, 0]
    margin = top2[:, 0] - top2[:, 1]
    entropy = -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum(dim=-1)
    return confidence, margin, entropy


def selector_routing_signals(selector_logits, temperature, tau, delta, risky_label_ids):
    probabilities = calibrated_probabilities(selector_logits, temperature)
    confidence, margin, entropy = probability_statistics(probabilities)
    predictions = probabilities.argmax(dim=-1)
    confidence_failure = confidence < tau
    margin_failure = margin < delta
    risky_class_trigger = predicted_risky_mask(predictions, risky_label_ids)
    route_to_fusion = confidence_failure | margin_failure | risky_class_trigger
    return {
        "probabilities": probabilities,
        "predictions": predictions,
        "confidence": confidence,
        "margin": margin,
        "entropy": entropy,
        "confidence_failure": confidence_failure,
        "margin_failure": margin_failure,
        "risky_class_trigger": risky_class_trigger,
        "route_to_fusion": route_to_fusion,
    }


@torch.no_grad()
def collect_dual_expert_logits(model, data_loader, device):
    model.eval()
    selector_logits_all = []
    roberta_logits_all = []
    labels_all = []
    for batch in data_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        h_electra = model.encode_electra(
            batch["electra_input_ids"],
            batch["electra_attention_mask"],
            batch.get("electra_token_type_ids"),
        )
        h_roberta = model.encode_roberta(
            batch["roberta_input_ids"],
            batch["roberta_attention_mask"],
        )
        selector_logits_all.append(
            model.selector_logits_from_representation(h_electra).detach().cpu()
        )
        roberta_logits_all.append(
            model.roberta_logits_from_representation(h_roberta).detach().cpu()
        )
        labels_all.append(batch["labels"].detach().cpu())
    return (
        torch.cat(selector_logits_all, dim=0),
        torch.cat(roberta_logits_all, dim=0),
        torch.cat(labels_all, dim=0),
    )


def fit_temperature_from_logits(logits, labels, device):
    logits = logits.detach().to(device)
    labels = labels.detach().to(device)
    nll_before = nn.functional.cross_entropy(logits, labels).item()
    log_temperature = nn.Parameter(torch.zeros(1, device=device))
    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=TEMPERATURE_SCALING_MAX_ITER,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)
        temperature = log_temperature.exp().clamp(0.05, 10.0)
        loss = nn.functional.cross_entropy(logits / temperature, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    temperature = float(log_temperature.detach().exp().clamp(0.05, 10.0).item())
    nll_after = nn.functional.cross_entropy(logits / temperature, labels).item()
    return temperature, nll_before, nll_after


def evaluate_cached_candidate(
    selector_logits,
    roberta_logits,
    labels,
    class_weights,
    selector_temperature,
    roberta_temperature,
    alpha,
    tau,
    delta,
    risky_label_ids,
):
    routing = selector_routing_signals(
        selector_logits,
        selector_temperature,
        tau,
        delta,
        risky_label_ids,
    )
    fused_probs = late_fusion_probabilities(
        selector_logits,
        roberta_logits,
        selector_temperature,
        roberta_temperature,
        alpha,
    )
    final_probs = routing["probabilities"].clone()
    final_probs[routing["route_to_fusion"]] = fused_probs[routing["route_to_fusion"]]
    preds = final_probs.argmax(dim=-1).cpu().numpy()
    label_array = labels.cpu().numpy()
    route = routing["route_to_fusion"].cpu().numpy().astype(bool)
    metrics = compute_metrics_from_arrays(label_array, preds)
    metrics["loss"] = nn.functional.nll_loss(
        torch.log(final_probs.clamp_min(1e-12)),
        labels,
        weight=class_weights,
    ).item()
    metrics["route_to_fusion_rate"] = float(route.mean())
    metrics["electra_only_rate"] = 1.0 - metrics["route_to_fusion_rate"]
    metrics["average_active_encoder_passes"] = 1.0 + metrics["route_to_fusion_rate"]
    metrics["mean_confidence"] = float(routing["confidence"].mean().item())
    metrics["mean_margin"] = float(routing["margin"].mean().item())
    metrics["mean_entropy"] = float(routing["entropy"].mean().item())
    metrics["confidence_route_reason_rate"] = float(
        routing["confidence_failure"].float().mean().item()
    )
    metrics["margin_route_reason_rate"] = float(
        routing["margin_failure"].float().mean().item()
    )
    metrics["risky_class_route_reason_rate"] = float(
        routing["risky_class_trigger"].float().mean().item()
    )
    metrics["alpha"] = float(alpha)
    metrics["tau"] = float(tau)
    metrics["delta"] = float(delta)
    return metrics


def tune_serf_thresholds(
    selector_logits,
    roberta_logits,
    labels,
    class_weights,
    selector_temperature,
    roberta_temperature,
):
    rows = []
    for risky_policy, risky_label_ids in RISKY_POLICY_CANDIDATES.items():
        for alpha in ALPHA_VALUES:
            for tau in TAU_VALUES:
                for delta in DELTA_VALUES:
                    metrics = evaluate_cached_candidate(
                        selector_logits,
                        roberta_logits,
                        labels,
                        class_weights,
                        selector_temperature,
                        roberta_temperature,
                        alpha,
                        tau,
                        delta,
                        risky_label_ids,
                    )
                    metrics["risky_policy"] = risky_policy
                    rows.append(metrics)

    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.sort_values(
        ["macro_F1", "route_to_fusion_rate", "accuracy"],
        ascending=[False, True, False],
    ).iloc[0].to_dict()
    return threshold_df, best_row


@torch.no_grad()
def forward_serf_batch(
    model,
    batch,
    mode,
    tau,
    delta,
    risky_label_ids,
    alpha,
    selector_temperature,
    roberta_temperature,
):
    supported_modes = {"selector_only", "roberta_only", "always_fusion", "selective"}
    if mode not in supported_modes:
        raise ValueError(f"Unsupported inference mode: {mode}")

    batch_size = batch["labels"].shape[0]
    if mode == "roberta_only":
        h_roberta = model.encode_roberta(
            batch["roberta_input_ids"],
            batch["roberta_attention_mask"],
        )
        roberta_logits = model.roberta_logits_from_representation(h_roberta)
        probabilities = calibrated_probabilities(roberta_logits, roberta_temperature)
        confidence, margin, entropy = probability_statistics(probabilities)
        false_mask = torch.zeros(batch_size, dtype=torch.bool, device=roberta_logits.device)
        return {
            "logits": torch.log(probabilities.clamp_min(1e-12)),
            "route_to_fusion": false_mask,
            "active_encoder_passes": torch.ones(batch_size, device=roberta_logits.device),
            "confidence": confidence,
            "margin": margin,
            "entropy": entropy,
            "confidence_failure": false_mask,
            "margin_failure": false_mask,
            "risky_class_trigger": false_mask,
        }

    h_electra = model.encode_electra(
        batch["electra_input_ids"],
        batch["electra_attention_mask"],
        batch.get("electra_token_type_ids"),
    )
    selector_logits = model.selector_logits_from_representation(h_electra)
    routing = selector_routing_signals(
        selector_logits,
        selector_temperature,
        tau,
        delta,
        risky_label_ids,
    )

    if mode == "selector_only":
        route_to_fusion = torch.zeros_like(routing["predictions"], dtype=torch.bool)
    elif mode == "always_fusion":
        route_to_fusion = torch.ones_like(routing["predictions"], dtype=torch.bool)
    else:
        route_to_fusion = routing["route_to_fusion"]

    final_probs = routing["probabilities"].clone()
    if route_to_fusion.any():
        h_roberta = model.encode_roberta(
            batch["roberta_input_ids"][route_to_fusion],
            batch["roberta_attention_mask"][route_to_fusion],
        )
        roberta_logits = model.roberta_logits_from_representation(h_roberta)
        fused_probs = late_fusion_probabilities(
            selector_logits[route_to_fusion],
            roberta_logits,
            selector_temperature,
            roberta_temperature,
            alpha,
        )
        final_probs[route_to_fusion] = fused_probs

    return {
        **routing,
        "logits": torch.log(final_probs.clamp_min(1e-12)),
        "route_to_fusion": route_to_fusion,
        "active_encoder_passes": 1.0 + route_to_fusion.float(),
    }


@torch.no_grad()
def evaluate_serf_mode(
    model,
    data_loader,
    device,
    mode,
    tau,
    delta,
    risky_label_ids,
    alpha,
    selector_temperature,
    roberta_temperature,
):
    model.eval()
    all_labels = []
    all_preds = []
    all_route_to_fusion = []
    all_active_passes = []
    all_confidence = []
    all_margin = []
    all_entropy = []
    all_confidence_failure = []
    all_margin_failure = []
    all_risky_class_trigger = []
    total_loss = 0.0

    for batch in data_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = forward_serf_batch(
            model,
            batch,
            mode,
            tau,
            delta,
            risky_label_ids,
            alpha,
            selector_temperature,
            roberta_temperature,
        )
        labels = batch["labels"]
        logits = outputs["logits"]
        preds = logits.argmax(dim=-1)
        total_loss += nn.functional.nll_loss(
            logits,
            labels,
            weight=model.class_weights,
        ).item()
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_route_to_fusion.extend(outputs["route_to_fusion"].detach().cpu().numpy())
        all_active_passes.extend(outputs["active_encoder_passes"].detach().cpu().numpy())
        all_confidence.extend(outputs["confidence"].detach().cpu().numpy())
        all_margin.extend(outputs["margin"].detach().cpu().numpy())
        all_entropy.extend(outputs["entropy"].detach().cpu().numpy())
        all_confidence_failure.extend(outputs["confidence_failure"].detach().cpu().numpy())
        all_margin_failure.extend(outputs["margin_failure"].detach().cpu().numpy())
        all_risky_class_trigger.extend(outputs["risky_class_trigger"].detach().cpu().numpy())

    labels = np.asarray(all_labels)
    preds = np.asarray(all_preds)
    route_to_fusion = np.asarray(all_route_to_fusion, dtype=bool)
    metrics = compute_metrics_from_arrays(labels, preds)
    metrics["loss"] = total_loss / max(1, len(data_loader))
    metrics["route_to_fusion_rate"] = float(route_to_fusion.mean())
    metrics["electra_only_rate"] = 1.0 - metrics["route_to_fusion_rate"]
    metrics["average_active_encoder_passes"] = float(np.mean(all_active_passes))
    metrics["mean_confidence"] = float(np.mean(all_confidence))
    metrics["mean_margin"] = float(np.mean(all_margin))
    metrics["mean_entropy"] = float(np.mean(all_entropy))
    metrics["confidence_route_reason_rate"] = float(np.mean(all_confidence_failure))
    metrics["margin_route_reason_rate"] = float(np.mean(all_margin_failure))
    metrics["risky_class_route_reason_rate"] = float(np.mean(all_risky_class_trigger))
    metrics["alpha"] = float(alpha)
    metrics["tau"] = float(tau)
    metrics["delta"] = float(delta)
    metrics["inference_mode"] = mode
    return metrics, labels, preds


def evaluate_selective_routing(
    model,
    data_loader,
    device,
    tau,
    delta,
    risky_label_ids,
    alpha,
    selector_temperature,
    roberta_temperature,
):
    return evaluate_serf_mode(
        model,
        data_loader,
        device,
        mode="selective",
        tau=tau,
        delta=delta,
        risky_label_ids=risky_label_ids,
        alpha=alpha,
        selector_temperature=selector_temperature,
        roberta_temperature=roberta_temperature,
    )


def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


@torch.no_grad()
def diagnose_serf_routing(
    model,
    data_loader,
    device,
    tau,
    delta,
    risky_label_ids,
    alpha,
    selector_temperature,
    roberta_temperature,
):
    selector_logits, roberta_logits, labels = collect_dual_expert_logits(
        model,
        data_loader,
        device,
    )
    routing = selector_routing_signals(
        selector_logits,
        selector_temperature,
        tau,
        delta,
        risky_label_ids,
    )
    roberta_probs = calibrated_probabilities(roberta_logits, roberta_temperature)
    fused_probs = late_fusion_probabilities(
        selector_logits,
        roberta_logits,
        selector_temperature,
        roberta_temperature,
        alpha,
    )
    selector_preds = routing["predictions"].numpy()
    roberta_preds = roberta_probs.argmax(dim=-1).numpy()
    fused_preds = fused_probs.argmax(dim=-1).numpy()
    label_array = labels.numpy()
    route = routing["route_to_fusion"].numpy().astype(bool)
    selector_wrong = selector_preds != label_array
    selector_correct = ~selector_wrong
    roberta_correct = roberta_preds == label_array
    fused_correct = fused_preds == label_array
    fused_wrong = ~fused_correct
    captured_errors = route & selector_wrong
    corrected = captured_errors & fused_correct
    degraded = route & selector_correct & fused_wrong

    oracle_preds = selector_preds.copy()
    oracle_mask = selector_wrong & fused_correct
    oracle_preds[oracle_mask] = fused_preds[oracle_mask]
    oracle_metrics = compute_metrics_from_arrays(label_array, oracle_preds)

    return {
        "selector_error_rate": float(selector_wrong.mean()),
        "roberta_error_rate": float((roberta_preds != label_array).mean()),
        "expert_disagreement_rate": float((selector_preds != roberta_preds).mean()),
        "error_capture_rate": safe_ratio(captured_errors.sum(), selector_wrong.sum()),
        "route_precision_for_selector_errors": safe_ratio(captured_errors.sum(), route.sum()),
        "routed_correction_rate": safe_ratio(corrected.sum(), captured_errors.sum()),
        "routed_degradation_rate": safe_ratio(
            degraded.sum(),
            (route & selector_correct).sum(),
        ),
        "roberta_correction_potential": safe_ratio(
            (selector_wrong & roberta_correct).sum(),
            selector_wrong.sum(),
        ),
        "late_fusion_correction_potential": safe_ratio(
            (selector_wrong & fused_correct).sum(),
            selector_wrong.sum(),
        ),
        "oracle_router_macro_F1": oracle_metrics["macro_F1"],
        "corrected_sample_count": int(corrected.sum()),
        "degraded_sample_count": int(degraded.sum()),
        "net_corrected_samples": int(corrected.sum() - degraded.sum()),
    }


@torch.no_grad()
def benchmark_serf_mode(
    model,
    data_loader,
    device,
    mode,
    tau,
    delta,
    risky_label_ids,
    alpha,
    selector_temperature,
    roberta_temperature,
):
    model.eval()

    for _ in range(BENCHMARK_WARMUP_ROUNDS):
        for batch in data_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            forward_serf_batch(
                model,
                batch,
                mode,
                tau,
                delta,
                risky_label_ids,
                alpha,
                selector_temperature,
                roberta_temperature,
            )
    synchronize_device(device)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    repeat_seconds = []
    for _ in range(BENCHMARK_REPEATS):
        synchronize_device(device)
        start = time.perf_counter()
        for batch in data_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            forward_serf_batch(
                model,
                batch,
                mode,
                tau,
                delta,
                risky_label_ids,
                alpha,
                selector_temperature,
                roberta_temperature,
            )
        synchronize_device(device)
        repeat_seconds.append(time.perf_counter() - start)

    sample_count = len(data_loader.dataset)
    per_sample_ms = np.asarray(repeat_seconds) * 1000.0 / sample_count
    total_samples = sample_count * BENCHMARK_REPEATS
    total_seconds = float(np.sum(repeat_seconds))
    peak_memory_mb = (
        torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        if device.type == "cuda"
        else np.nan
    )
    return {
        "pretokenized_latency_ms_per_sample": float(np.mean(per_sample_ms)),
        "p50_latency_ms_per_sample": float(np.percentile(per_sample_ms, 50)),
        "p95_latency_ms_per_sample": float(np.percentile(per_sample_ms, 95)),
        "repeat_latency_ms_per_sample": per_sample_ms.tolist(),
        "throughput_samples_per_second": total_samples / total_seconds,
        "peak_gpu_memory_mb": float(peak_memory_mb),
        "benchmark_repeats": BENCHMARK_REPEATS,
        "benchmark_warmup_rounds": BENCHMARK_WARMUP_ROUNDS,
        "tokenization_included": False,
        "benchmark_scope": "pretokenized test-loader inference including device transfer",
        "inference_precision": "FP32",
    }


def plot_serf_confusion_matrix(labels, preds, run_name):
    print(classification_report(
        labels,
        preds,
        target_names=label_encoder.classes_,
        zero_division=0,
    ))
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        cmap="Blues",
    )
    plt.title(f"Confusion Matrix - {run_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def run_serf_experiment(train_frame, experiment_name, run_seed=None):
    if run_seed is None:
        run_seed = RANDOM_STATE

    total_start = time.perf_counter()
    class_weights = compute_class_weights(train_frame)

    set_all_seeds(run_seed)
    selector_train_loader, selector_val_loader, _ = make_loaders(
        train_frame,
        seed=run_seed,
    )
    model = ElectraRobertaLateFusionSERF(
        num_labels=num_labels,
        class_weights=class_weights,
    )
    model.enable_memory_saving()
    model.configure_selector_stage()
    selector_start = time.perf_counter()
    model, device = train_model(
        model,
        selector_train_loader,
        selector_val_loader,
        run_name=f"ELECTRA selector ({experiment_name}, seed={run_seed})",
    )
    selector_train_seconds = time.perf_counter() - selector_start
    selector_training_metadata = dict(model.training_metadata)
    _, stage1_val_labels, stage1_selector_preds = evaluate_model(
        model,
        selector_val_loader,
        device,
    )

    # Stage 2 trains only the independent RoBERTa expert; ELECTRA is unchanged.
    set_all_seeds(run_seed)
    roberta_train_loader, roberta_val_loader, roberta_test_loader = make_loaders(
        train_frame,
        seed=run_seed,
    )
    model.configure_roberta_stage()
    roberta_start = time.perf_counter()
    model, device = train_model(
        model,
        roberta_train_loader,
        roberta_val_loader,
        run_name=f"Independent RoBERTa expert ({experiment_name}, seed={run_seed})",
    )
    roberta_train_seconds = time.perf_counter() - roberta_start
    roberta_training_metadata = dict(model.training_metadata)
    model.configure_inference()

    val_selector_logits, val_roberta_logits, val_labels = collect_dual_expert_logits(
        model,
        roberta_val_loader,
        device,
    )
    selector_temperature, selector_nll_before, selector_nll_after = (
        fit_temperature_from_logits(val_selector_logits, val_labels, device)
    )
    roberta_temperature, roberta_nll_before, roberta_nll_after = (
        fit_temperature_from_logits(val_roberta_logits, val_labels, device)
    )

    threshold_start = time.perf_counter()
    threshold_df, best_threshold = tune_serf_thresholds(
        val_selector_logits,
        val_roberta_logits,
        val_labels,
        class_weights,
        selector_temperature,
        roberta_temperature,
    )
    threshold_search_seconds = time.perf_counter() - threshold_start
    selected_risky_policy = str(best_threshold["risky_policy"])
    selected_risky_label_ids = RISKY_POLICY_CANDIDATES[selected_risky_policy]

    synchronize_device(device)
    test_start = time.perf_counter()
    test_metrics, test_labels, test_preds = evaluate_selective_routing(
        model,
        roberta_test_loader,
        device,
        tau=best_threshold["tau"],
        delta=best_threshold["delta"],
        risky_label_ids=selected_risky_label_ids,
        alpha=best_threshold["alpha"],
        selector_temperature=selector_temperature,
        roberta_temperature=roberta_temperature,
    )
    synchronize_device(device)
    routed_test_seconds = time.perf_counter() - test_start

    mode_rows = []
    for mode in ("selector_only", "roberta_only", "always_fusion", "selective"):
        mode_metrics, _, mode_preds = evaluate_serf_mode(
            model,
            roberta_test_loader,
            device,
            mode=mode,
            tau=best_threshold["tau"],
            delta=best_threshold["delta"],
            risky_label_ids=selected_risky_label_ids,
            alpha=best_threshold["alpha"],
            selector_temperature=selector_temperature,
            roberta_temperature=roberta_temperature,
        )
        benchmark_metrics = benchmark_serf_mode(
            model,
            roberta_test_loader,
            device,
            mode=mode,
            tau=best_threshold["tau"],
            delta=best_threshold["delta"],
            risky_label_ids=selected_risky_label_ids,
            alpha=best_threshold["alpha"],
            selector_temperature=selector_temperature,
            roberta_temperature=roberta_temperature,
        )
        mode_rows.append({
            "model": "Late-fusion ELECTRA-RoBERTa SERF",
            "experiment": experiment_name,
            "seed": run_seed,
            "mode": mode,
            "selected_tau": best_threshold["tau"],
            "selected_delta": best_threshold["delta"],
            "selected_alpha": best_threshold["alpha"],
            "selected_risky_policy": selected_risky_policy,
            **mode_metrics,
            **benchmark_metrics,
        })

    mode_results_df = pd.DataFrame(mode_rows)
    mode_lookup = {row["mode"]: row for row in mode_rows}
    selector_only_result = mode_lookup["selector_only"]
    roberta_only_result = mode_lookup["roberta_only"]
    always_fusion_result = mode_lookup["always_fusion"]
    selective_benchmark = mode_lookup["selective"]
    post_stage2_val_selector_preds = val_selector_logits.argmax(dim=-1).numpy()
    selector_prediction_drift_rate = float(
        np.mean(np.asarray(stage1_selector_preds) != post_stage2_val_selector_preds)
    )
    if not np.array_equal(stage1_val_labels, val_labels.numpy()):
        raise RuntimeError("Validation-label order changed between training stages")

    routing_diagnostics = diagnose_serf_routing(
        model,
        roberta_test_loader,
        device,
        tau=best_threshold["tau"],
        delta=best_threshold["delta"],
        risky_label_ids=selected_risky_label_ids,
        alpha=best_threshold["alpha"],
        selector_temperature=selector_temperature,
        roberta_temperature=roberta_temperature,
    )
    routing_diagnostics["selector_prediction_drift_rate"] = selector_prediction_drift_rate
    training_and_selection_seconds = time.perf_counter() - total_start

    run_name = f"Late-fusion ELECTRA-RoBERTa SERF ({experiment_name}, seed={run_seed})"
    print(
        f"{run_name} | selected_tau={best_threshold['tau']:.2f} "
        f"| selected_delta={best_threshold['delta']:.2f} "
        f"| selected_alpha={best_threshold['alpha']:.2f} "
        f"| risky_policy={selected_risky_policy} "
        f"| val_macro_F1={best_threshold['macro_F1']:.4f} "
        f"| test_macro_F1={test_metrics['macro_F1']:.4f} "
        f"| route_to_fusion_rate={test_metrics['route_to_fusion_rate']:.4f}"
    )
    print("Routing and expert diagnostics:", routing_diagnostics)
    plot_serf_confusion_matrix(test_labels, test_preds, run_name)

    result = {
        "model": f"Late-fusion ELECTRA-RoBERTa SERF ({experiment_name})",
        "seed": run_seed,
        "selected_tau": best_threshold["tau"],
        "selected_delta": best_threshold["delta"],
        "selected_alpha": best_threshold["alpha"],
        "selected_risky_policy": selected_risky_policy,
        "val_accuracy": best_threshold["accuracy"],
        "val_macro_F1": best_threshold["macro_F1"],
        "val_route_to_fusion_rate": best_threshold["route_to_fusion_rate"],
        **test_metrics,
        **routing_diagnostics,
        "selector_only_macro_F1": selector_only_result["macro_F1"],
        "roberta_only_macro_F1": roberta_only_result["macro_F1"],
        "always_fusion_macro_F1": always_fusion_result["macro_F1"],
        "selector_temperature": selector_temperature,
        "roberta_temperature": roberta_temperature,
        "selector_calibration_nll_before": selector_nll_before,
        "selector_calibration_nll_after": selector_nll_after,
        "roberta_calibration_nll_before": roberta_nll_before,
        "roberta_calibration_nll_after": roberta_nll_after,
        "pretokenized_latency_ms_per_sample": selective_benchmark["pretokenized_latency_ms_per_sample"],
        "p50_latency_ms_per_sample": selective_benchmark["p50_latency_ms_per_sample"],
        "p95_latency_ms_per_sample": selective_benchmark["p95_latency_ms_per_sample"],
        "throughput_samples_per_second": selective_benchmark["throughput_samples_per_second"],
        "peak_gpu_memory_mb": selective_benchmark["peak_gpu_memory_mb"],
        "selector_only_latency_ms_per_sample": selector_only_result["pretokenized_latency_ms_per_sample"],
        "roberta_only_latency_ms_per_sample": roberta_only_result["pretokenized_latency_ms_per_sample"],
        "always_fusion_latency_ms_per_sample": always_fusion_result["pretokenized_latency_ms_per_sample"],
        "parameter_count": sum(param.numel() for param in model.parameters()),
        "selector_best_epoch": selector_training_metadata["best_epoch"],
        "selector_epochs_run": selector_training_metadata["epochs_run"],
        "roberta_best_epoch": roberta_training_metadata["best_epoch"],
        "roberta_epochs_run": roberta_training_metadata["epochs_run"],
        "selector_train_seconds": selector_train_seconds,
        "roberta_train_seconds": roberta_train_seconds,
        "threshold_search_seconds": threshold_search_seconds,
        "routed_test_seconds": routed_test_seconds,
        "training_and_selection_seconds": training_and_selection_seconds,
        "runtime_seconds": time.perf_counter() - total_start,
    }

    threshold_df = threshold_df.copy()
    threshold_df["model"] = f"Late-fusion ELECTRA-RoBERTa SERF ({experiment_name})"
    threshold_df["seed"] = run_seed
    threshold_df["selector_temperature"] = selector_temperature
    threshold_df["roberta_temperature"] = roberta_temperature
    threshold_df["selected"] = (
        (threshold_df["tau"] == best_threshold["tau"])
        & (threshold_df["delta"] == best_threshold["delta"])
        & (threshold_df["alpha"] == best_threshold["alpha"])
        & (threshold_df["risky_policy"] == selected_risky_policy)
    )
    return result, threshold_df, mode_results_df


In [ ]:
# --- Late-fusion ELECTRA-RoBERTa SERF WITHOUT augmentation: 3 fixed seeds ---
# SPLIT_SEED=42 fixes the stratified split; only training/model seeds vary.
serf_no_aug_results = []
serf_no_aug_threshold_rows = []
serf_no_aug_mode_rows = []

for run_seed in TRAIN_SEEDS:
    seed_result, seed_threshold_df, seed_mode_df = run_serf_experiment(
        train_df,
        "without augmentation",
        run_seed=run_seed,
    )
    serf_no_aug_results.append(seed_result)
    serf_no_aug_threshold_rows.append(seed_threshold_df)
    serf_no_aug_mode_rows.append(seed_mode_df)

serf_no_aug_results_df = pd.DataFrame(serf_no_aug_results)
serf_no_aug_threshold_results_df = pd.concat(serf_no_aug_threshold_rows, ignore_index=True)
serf_no_aug_mode_results_df = pd.concat(serf_no_aug_mode_rows, ignore_index=True)
serf_no_aug_summary = summarize_seed_results(serf_no_aug_results_df, SERF_METRIC_COLUMNS)

print("Late-fusion SERF without augmentation validation search:")
display(serf_no_aug_threshold_results_df)
print("Late-fusion SERF without augmentation matched inference modes:")
display(serf_no_aug_mode_results_df)
print("Late-fusion SERF without augmentation seed-level test results:")
display(serf_no_aug_results_df)
print("Late-fusion SERF without augmentation mean +/- std:")
display(serf_no_aug_summary)

print("Recorded runtime environment:")
display(pd.DataFrame([collect_runtime_environment()]))
